In [ ]:
# --- Modelo Preditivo. Baseline simples, utilizando média móvel: 
# O modelo utiliza a média móvel para prever os próximos valores assumindo que o valor continuará constante ---
# Objetivo: Prever o tempo necessário para repor estoque de um certo produto, através da média móvel.
# Período selecionado para calcular a média: Última semana de 2023 (25-12-2023 à 31-12-2023).
# Período para aplicar a previsão: Primeira semana de Janeiro de 2024.

import pandas as pd
from pathlib import Path

# Definição dos diretórios para leitura dos dados
current_dir = Path.cwd()
root_dir = current_dir.parent
raw_data_dir = root_dir / 'data' / '1_raw'
processed_data_dir = root_dir / 'data' / '2_processed'
# Dados da tabela de vendas
df_vendas = pd.read_csv(raw_data_dir/"vendas_2023_2024.csv")
# Calendário com range 01-01-2023 à 31-12-2024
df_dias = pd.read_csv(processed_data_dir/'calendar_2023_2024.csv', sep=';')

In [56]:
# Renomeação da coluna data do Dataframe de produtos
df_dias.rename(columns={'data':'sale_date'}, inplace=True)

In [57]:
# Junção das tabelas por left join pelo calendário, para encontrar possíveis valores nulos
df_final = pd.merge(df_dias,df_vendas,on='sale_date', how='left')
# Exibição dos valores nulos
df_final.isnull().sum()

sale_date     0
ano           0
mes           0
nome_mes      0
dia_semana    0
id            8
id_client     8
id_product    8
qtd           8
total         8
dtype: int64

In [58]:
# Reatribuição e tratamento dos dados
df_final = df_final.fillna(0)
# Checagem dos valores nulos após tratamento
df_final.isnull().sum()

sale_date     0
ano           0
mes           0
nome_mes      0
dia_semana    0
id            0
id_client     0
id_product    0
qtd           0
total         0
dtype: int64

In [59]:
# Agrupamento de dados por sale_date e id_product, somando as qtd de produtos dos dados agrupados, reset_index para formatar a tabela novamente
df_diario = df_final.groupby(['sale_date', 'id_product'])['qtd'].sum().reset_index()

In [60]:
# Determinando limite de dados a ser analisado
ultima_semana_2023 = df_diario[
    (df_diario['sale_date'] >= '2023-12-25') & 
    (df_diario['sale_date'] < '2024-01-01')
]

In [61]:
# Cálculo do baseline através da média móvel de qtd por produtos, utilizando a última semana de 2023 como referência
baseline_produtos = ultima_semana_2023.groupby('id_product')['qtd'].mean().reset_index()
# Renomeação da coluna qtd para média diária
baseline_produtos.rename(columns={'qtd':'media_diaria'}, inplace=True)

In [62]:
# Criação do Dataframe para receber as datas de Janeiro de 2024
periodo_janeiro = pd.date_range(start='2024-01-01', end='2024-01-31')
# Alimentação para o Dataframe através das sale_dates para o Dataframe df_calendario_janeiro
df_calendario_janeiro = pd.DataFrame({'sale_date':periodo_janeiro})

In [63]:
# Criação de uma coluna['key] = 1 para juntar os dois Dataframes
df_calendario_janeiro['key'] = 1
baseline_produtos['key'] = 1

In [64]:
# Junção dos Dataframes e dropando a coluna key, pois não será mais necessário
previsao_diaria = pd.merge(df_calendario_janeiro, baseline_produtos, on='key').drop('key', axis=1)

In [65]:
# Formatação para a média diária se tornar um valor inteiro, não é possível vender 1,5 de algum produto
previsao_diaria['previsao_venda_diaria'] = previsao_diaria['media_diaria'].round(0)
previsao_diaria.head()

,sale_date,id_product,media_diaria,previsao_venda_diaria
0,2024-01-01,3.0,10.0,10.0
1,2024-01-01,7.0,10.0,10.0
2,2024-01-01,10.0,10.0,10.0
3,2024-01-01,11.0,19.0,19.0
4,2024-01-01,13.0,3.0,3.0


In [ ]:
# Leitura do arquivo produtos e renomeação da tabela code para id_product
df_produtos = pd.read_csv(processed_data_dir/'produtos_tratados.csv', sep=';')
df_produtos.rename(columns={'code':'id_product'},inplace=True)
df_produtos.head()

,product_name,price,id_product,actual_category
0,Transponder AIS Maré Magnum,33122.52,1,eletrônicos
1,Transponder Furuno Marlin,13998.15,2,eletrônicos
2,Radar Furuno Pulse Leviathan,9024.19,3,eletrônicos
3,Rádio AIS Hydro Tidal Zen,3381.88,4,eletrônicos
4,Piloto Automático Furuno Storm,23669.01,5,eletrônicos


In [67]:
# Junção dos Dataframes com a média prevista e produtos
df_final = pd.merge(previsao_diaria,df_produtos,on='id_product',how='left')
df_final.rename(columns={'name':'product_name'}, inplace=True)

In [68]:
# Definição de colunas a serem visualizadas na consulta e exibição
colunas_finais = ['sale_date', 'id_product', 'product_name', 'previsao_venda_diaria']
df_relatorio = df_final[colunas_finais]

In [ ]:
# Criação do diretório para salvar o relatório e exportação do Dataframe para .csv
report_baseline_dir = root_dir / 'report' / 'estoque'
report_baseline_dir.mkdir(parents=True,exist_ok=True)
df_relatorio.to_csv(report_baseline_dir/'relatorio_comprar_estoque.csv', index=False, sep=';')

In [71]:
# Definição para analisar um produto em específico e delimitando a primeira semana de 2024
venda_semanal_motor155hp = df_relatorio[
    (df_relatorio['id_product'] == 54.0) & 
    (df_relatorio['sale_date'] >= '2024-01-01') & 
    (df_relatorio['sale_date'] <= '2024-01-07')
]
# Atribuição e visualização da previsão total para o Motor 155 HP
total_previsao = venda_semanal_motor155hp['previsao_venda_diaria'].sum()
print(f"Previsão total para semana de janeiro: {int(total_previsao)}")

Previsão total para semana de janeiro: 0
